In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.quantization
from transformers import AutoModelForAudioClassification,ASTFeatureExtractor



In [ ]:
# 1. Load your fine-tuned model
model_path = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
model = AutoModelForAudioClassification.from_pretrained(model_path)
model.eval()


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=T

In [ ]:

# 2. Assign Quantization Configuration
# 'fbgemm' is for x86 CPUs (like your Dell Inspiron); 'qnnpack' is for ARM (mobile).
model.qconfig = torch.quantization.get_default_qconfig('fbgemm')

# for mobile devices
# model.qconfig = torch.quantization.get_default_qconfig('qnnpack')



In [ ]:
# 3. Prepare the model for quantization
# This inserts "observers" that watch the data during calibration
model_prepared = torch.quantization.prepare(model)


/tmp/ipykernel_9907/1347395045.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = torch.quantization.prepare(model)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of 

In [ ]:
feature_extractor = ASTFeatureExtractor.from_pretrained(model_path)

In [ ]:
from datasets import load_from_disk
dataset_path = "/content/drive/MyDrive/Urban8KAudioFiles/processed_train"
test_dataset = load_from_disk(dataset_path)

In [ ]:
print(test_dataset.column_names)

['input_values', 'labels']


In [ ]:
import torch

# 3. Calibration Step
print("Calibrating for Quantization using 'input_values'...")
# model_prepared.to('cpu') # Ensure model is on CPU for quantization
model_prepared.eval()

with torch.no_grad():
    for i in range(20):
        # 1. Get the pre-processed spectrogram patches
        # We wrap it in a list [ ] and convert to tensor to create the batch dimension
        feat = torch.tensor([test_dataset[i]["input_values"]])

        # 2. Prepare the dictionary for the model
        inputs = {"input_values": feat}

        # 3. Forward pass for calibration
        model_prepared(**inputs)

print("Calibration successful! The observers have recorded the dynamic range.")

Calibrating for Quantization using 'input_values'...
Calibration successful! The observers have recorded the dynamic range.


In [ ]:
# 5. Convert to Quantized Model
model_int8 = torch.quantization.convert(model_prepared)



/tmp/ipykernel_9907/2398995763.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.convert(model_prepared)


In [ ]:
# 6. Save the Quantized Model
torch.save(model_int8.state_dict(), f"{model_path}/ast_model_int8_v2.pth")
print("Quantization Complete! Model size reduced by ~4x.")

Quantization Complete! Model size reduced by ~4x.


# Testing Quantization model

In [ ]:
# Force the model to the CPU
model_int8.to('cpu')

def evaluate_quantized_model(model, dataset):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i in range(len(dataset)):
            # Ensure the input tensor is explicitly on the CPU
            inputs = torch.tensor([dataset[i]["input_values"]]).to('cpu')

            logits = model(inputs).logits
            prediction = torch.argmax(logits, dim=-1).item()

            all_preds.append(prediction)
            all_labels.append(dataset[i]["labels"])
    # ... rest of your accuracy calculation

In [ ]:
evaluate_quantized_model(model_int8,test_dataset)

In [ ]:
import torch
import torchaudio
import time
from transformers import ASTFeatureExtractor, AutoModelForAudioClassification

# 1. Paths
model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
test_audio_path = "test_sound.wav" # Replace with your file

# 2. Load the base model directly to CPU
print("Loading model structure...")
model = AutoModelForAudioClassification.from_pretrained(model_dir)
model.to('cpu')
model.eval()

# 3. Apply Robust Dynamic Quantization
# We target the Linear layers (nn.Linear) which dominate the Transformer attention blocks
print("Applying Dynamic Quantization to Transformer Linear layers...")
model_int8 = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear}, # Target layers
    dtype=torch.qint8  # Quantize weights to INT8
)

# 4. Process the Audio File
feature_extractor = ASTFeatureExtractor.from_pretrained(model_dir)

# Load and resample to 16kHz
audio, sr = torchaudio.load(test_audio_path)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(audio)

# Convert to Mel Spectrogram patches
inputs = feature_extractor(audio.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to('cpu') for k, v in inputs.items()}

# 5. Run Error-Free Inference
print("Running inference on CPU...")
start_time = time.time()
with torch.no_grad():
    outputs = model_int8(inputs['input_values'])
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
latency = (time.time() - start_time) * 1000
urban8KSounds = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music"
  }
# 6. Map to Class Name
class_name = urban8KSounds.get(prediction, f"Unknown_Label_{prediction}")
print("\n--- Inference Results ---")
print(f"Prediction: {class_name}")
print(f"Inference Latency: {latency:.2f} ms")

Loading model structure...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Applying Dynamic Quantization to Transformer Linear layers...


/tmp/ipykernel_4574/2693534951.py:19: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


Running inference on CPU...

--- Inference Results ---
Prediction: air_conditioner
Inference Latency: 3552.02 ms


In [ ]:
import torch
from transformers import AutoModelForAudioClassification

model_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = f"{model_dir}/ast_dynamic_int8.pth"
test_audio_path = "test1.wav" # Replace with your file
print("1. Rebuilding base model architecture...")
# Load the unquantized architecture shell directly to CPU
base_model = AutoModelForAudioClassification.from_pretrained(model_dir)
base_model.to('cpu')

print("2. Instantiating dynamic quantization layers...")
# Dynamically quantize the shell to match the exact layer transformation of the saved weights
loaded_model_int8 = torch.quantization.quantize_dynamic(
    base_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("3. Loading quantized weights into the architecture...")
# Inject the saved INT8 weights into our structured model
loaded_model_int8.load_state_dict(torch.load(quantized_model_path, map_location='cpu'))
loaded_model_int8.eval()

print("Model is fully loaded and ready for inference!")

# 4. Process the Audio File
feature_extractor = ASTFeatureExtractor.from_pretrained(model_dir)

# Load and resample to 16kHz
audio, sr = torchaudio.load(test_audio_path)
if sr != 16000:
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(audio)

# Convert to Mel Spectrogram patches
inputs = feature_extractor(audio.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
inputs = {k: v.to('cpu') for k, v in inputs.items()}

# 5. Run Error-Free Inference
print("Running inference on CPU...")
start_time = time.time()
with torch.no_grad():
    outputs = loaded_model_int8(inputs['input_values'])
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
latency = (time.time() - start_time) * 1000
urban8KSounds = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music"
  }
# 6. Map to Class Name
class_name = urban8KSounds.get(prediction, f"Unknown_Label_{prediction}")
print("\n--- Inference Results ---")
print(f"Prediction: {class_name}")
print(f"Inference Latency: {latency:.2f} ms")

1. Rebuilding base model architecture...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

2. Instantiating dynamic quantization layers...


/tmp/ipykernel_4574/2486576046.py:14: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  loaded_model_int8 = torch.quantization.quantize_dynamic(


3. Loading quantized weights into the architecture...
Model is fully loaded and ready for inference!
Running inference on CPU...

--- Inference Results ---
Prediction: children_playing
Inference Latency: 4414.01 ms


# Saving Model

In [ ]:
import os

# Define your save path
save_dir = "/content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model"
quantized_model_path = os.path.join(save_dir, "ast_dynamic_int8.pth")

# Save only the weights (state_dict) of the quantized model
torch.save(model_int8.state_dict(), quantized_model_path)
print(f"Quantized model successfully saved to: {quantized_model_path}")

Quantized model successfully saved to: /content/drive/MyDrive/Urban8KAudioFiles/finetuned_ast_model/ast_dynamic_int8.pth


In [ ]:
original_size = os.path.getsize(f"{model_dir}/model.safetensors") / (1024 * 1024)
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024)

print(f"Original Model Size: {original_size:.2f} MB")
print(f"Quantized Model Size: {quantized_size:.2f} MB")
print(f"Storage Reduction: {((original_size - quantized_size) / original_size) * 100:.2f}%")

Original Model Size: 328.84 MB
Quantized Model Size: 85.94 MB
Storage Reduction: 73.87%
